In [1]:
import pandas as pd
import geopandas as gpd
import rioxarray
import numpy as np

In [2]:
deposits = pd.read_csv('../data/processed/az_porphyry_deposits.csv')
# geochem = pd.read_csv('../data/processed/geochemical_values.csv')
grid = gpd.read_file('../data/processed/grid_labels.geojson')
#geology = gpd.read_file('../data/raw/DI-08ver3/shapefiles/geopolys.shp')
mag_az = rioxarray.open_rasterio('../data/processed/mag_az.tif')

c:\Users\HP\miniconda3\envs\arizona-porphyry\Lib\site-packages\pyogrio\core.py:34: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


In [3]:

# --- Load raw data ---
best = pd.read_csv("../data/raw/ngdbsed-fUS04/bestvalue.txt", dtype=str, low_memory=False)
samples = pd.read_csv("../data/raw/ngdbsed-fUS04/main.txt", skiprows=1, header=None, dtype=str,low_memory=False)

# Rename main.txt columns (same as Phase 0)
cols = ["lab_id", "field_id", "job_id", "submitter",
        "date_collect", "date_submitted", "project_name",
        "state", "country",
        "lat_wgs84", "long_wgs84", "datum_orig",
        "lat_orig", "long_orig"] + [f"extra_{i}" for i in range(samples.shape[1] - 14)]
samples.columns = cols

# Filter to AZ
samples = samples[samples["state"].str.strip() == "AZ"].copy()
samples["lat"] = pd.to_numeric(samples["lat_wgs84"])
samples["lon"] = pd.to_numeric(samples["long_wgs84"])

# Filter to elements we care about
keep = ["CU", "MO", "AS", "PB", "ZN", "AU"]
best = best[best["species"].str.upper().isin(keep)].copy()

# --- KEY STEP: preserve the sign ---
best["value_signed"] = pd.to_numeric(best["qvalue"])
best["is_censored"] = best["value_signed"] < 0

# For censored: value = NaN, lod = abs(signed)
# For non-censored: value = signed, lod = NaN
best["value_clean"] = np.where(best["is_censored"], np.nan, best["value_signed"])
best["lod"] = np.where(best["is_censored"], best["value_signed"].abs(), np.nan)

print("Sanity check:")
print(best[["lab_id", "species", "qvalue", "value_signed", "is_censored", "value_clean", "lod"]].head(10))

Sanity check:
    lab_id species qvalue  value_signed  is_censored  value_clean    lod
1   LFP527      As   -200        -200.0         True          NaN  200.0
2   LFP527      Au    -10         -10.0         True          NaN   10.0
11  LFP527      Cu      7           7.0        False          7.0    NaN
16  LFP527      Mo     -5          -5.0         True          NaN    5.0
19  LFP527      Pb    -10         -10.0         True          NaN   10.0
29  LFP527      Zn   -200        -200.0         True          NaN  200.0
32  LFP528      As   -200        -200.0         True          NaN  200.0
33  LFP528      Au    -10         -10.0         True          NaN   10.0
42  LFP528      Cu     -5          -5.0         True          NaN    5.0
47  LFP528      Mo     -5          -5.0         True          NaN    5.0


In [4]:
# Three pivots — value, lod, censored
wide_value = best.pivot_table(
    index="lab_id", columns="species", values="value_clean", aggfunc="first"
)
wide_lod = best.pivot_table(
    index="lab_id", columns="species", values="lod", aggfunc="first"
).add_suffix("_lod")
wide_cens = best.pivot_table(
    index="lab_id", columns="species", values="is_censored", aggfunc="first"
).add_suffix("_censored")

# Merge all three
geochem = samples[["lab_id", "lat", "lon"]].merge(
    wide_value.join(wide_lod).join(wide_cens),
    on="lab_id", how="inner"
)

print("Shape:", geochem.shape)
print("\nSample rows:")
print(geochem[["Cu", "Cu_lod", "Cu_censored", "Mo", "Mo_lod", "Mo_censored"]].head(10))

Shape: (15052, 21)

Sample rows:
     Cu  Cu_lod Cu_censored    Mo  Mo_lod Mo_censored
0  16.0     NaN       False   NaN     2.0        True
1  17.0     NaN       False   NaN     2.0        True
2  22.0     NaN       False  2.00     NaN       False
3   5.0     NaN       False   NaN     2.0        True
4  15.0     NaN       False   NaN     2.0        True
5  94.0     NaN       False  2.50     NaN       False
6  34.4     NaN       False  1.10     NaN       False
7  62.9     NaN       False  1.10     NaN       False
8  17.9     NaN       False  0.36     NaN       False
9  21.4     NaN       False  0.44     NaN       False


In [5]:
# Impute censored as LOD/2, keep measured as-is
for el in ["Cu", "Mo", "As", "Pb", "Zn", "Au"]:
    geochem[f"{el}_imputed"] = geochem[el].copy()  # start with measured
    # Fill in censored with LOD/2
    censored_mask = geochem[f"{el}_censored"] == True
    geochem.loc[censored_mask, f"{el}_imputed"] = geochem.loc[censored_mask, f"{el}_lod"] / 2

# Verify
print(geochem[["Cu", "Cu_lod", "Cu_censored", "Cu_imputed"]].head(15))

      Cu  Cu_lod Cu_censored  Cu_imputed
0   16.0     NaN       False        16.0
1   17.0     NaN       False        17.0
2   22.0     NaN       False        22.0
3    5.0     NaN       False         5.0
4   15.0     NaN       False        15.0
5   94.0     NaN       False        94.0
6   34.4     NaN       False        34.4
7   62.9     NaN       False        62.9
8   17.9     NaN       False        17.9
9   21.4     NaN       False        21.4
10  39.8     NaN       False        39.8
11  23.5     NaN       False        23.5
12  36.6     NaN       False        36.6
13  43.2     NaN       False        43.2
14  56.6     NaN       False        56.6


In [6]:
elements = ["Cu", "Mo", "As", "Pb", "Zn", "Au"]

rows = []
for el in elements:
    measured = geochem[el].notna().sum()
    censored = (geochem[f"{el}_censored"] == True).sum()
    missing = geochem[el].isna().sum() - censored
    
    rows.append({
        "Element": el,
        "Measured": measured,
        "Censored": censored,
        "Missing": missing,
        "Total": len(geochem),
        "Pct_measured": round(100 * measured / len(geochem), 2),
        "Pct_censored_of_detected": round(100 * censored / (measured + censored), 2)
    })

coverage = pd.DataFrame(rows).set_index("Element")
print(coverage)

         Measured  Censored  Missing  Total  Pct_measured  \
Element                                                     
Cu          14606       162      284  15052         97.04   
Mo           2260     12508      284  15052         15.01   
As           2381     12431      240  15052         15.82   
Pb          14287       481      284  15052         94.92   
Zn           5236      9531      285  15052         34.79   
Au           1677     13126      249  15052         11.14   

         Pct_censored_of_detected  
Element                            
Cu                           1.10  
Mo                          84.70  
As                          83.93  
Pb                           3.26  
Zn                          64.54  
Au                          88.67  


In [7]:
# How many rows are fully NaN across all 6 elements?
all_nan = geochem[["Cu", "Mo", "As", "Pb", "Zn", "Au"]].isna().all(axis=1).sum()
print(f"Rows with all 6 elements NaN: {all_nan}")

# Look at one
if all_nan > 0:
    print("\nExample row with all NaN:")
    print(geochem[geochem[["Cu", "Mo", "As", "Pb", "Zn", "Au"]].isna().all(axis=1)].head(3))

Rows with all 6 elements NaN: 0


In [8]:
threshold = {
"Cu":1000,
"Mo":100,
"As":1000,
"Pb":1000,
"Zn":1000,
"Au":1000
}

thres_summary ={}
for element, thresh in threshold.items():
    col = geochem[f"{element}_imputed"]
    n_excess = (col > thresh).sum()
    percent_excess = 100* n_excess/ col.notna().sum()
    thres_summary[element] ={
        "Threshold": thresh,
        "Excess Count": int(n_excess), 
        "Percent Excess": round(percent_excess, 3)
    }

outlier_df = pd.DataFrame(thres_summary).T
print(outlier_df)

    Threshold  Excess Count  Percent Excess
Cu     1000.0          38.0           0.257
Mo      100.0          12.0           0.081
As     1000.0          10.0           0.068
Pb     1000.0          45.0           0.305
Zn     1000.0          62.0           0.420
Au     1000.0           0.0           0.000


In [9]:
# --- Build GeoDataFrames in a projected CRS (meters) ---
# 1. Outliers
cu_outliers = geochem[geochem["Cu"] > 1000].copy()
print(f"Cu outliers: {len(cu_outliers)}")

cu_outliers_gdf = gpd.GeoDataFrame(
    cu_outliers,
    geometry=gpd.points_from_xy(cu_outliers["lon"], cu_outliers["lat"]),
    crs="EPSG:4326"
).to_crs("EPSG:32612")

# 2. Deposits
deposits_gdf = gpd.GeoDataFrame(
    deposits,
    geometry=gpd.points_from_xy(deposits["longitude"], deposits["latitude"]),
    crs="EPSG:4326"
).to_crs("EPSG:32612")

Cu outliers: 38


In [10]:
# sjoin_nearest returns the nearest deposit (and distance in CRS units = meters)
outliers_with_nearest = gpd.sjoin_nearest(
    cu_outliers_gdf,
    deposits_gdf[["site_name", "geometry"]],
    how="left",
    distance_col="dist_to_deposit_m"
)

# How many are close?
bins = [0, 5000, 10000, 25000, 50000, 100000, float("inf")]
labels = ["<5km", "5-10km", "10-25km", "25-50km", "50-100km", ">100km"]
outliers_with_nearest["dist_bin"] = pd.cut(
    outliers_with_nearest["dist_to_deposit_m"], bins=bins, labels=labels
)

print(outliers_with_nearest["dist_bin"].value_counts().sort_index())

dist_bin
<5km         5
5-10km       4
10-25km     16
25-50km      7
50-100km     3
>100km       3
Name: count, dtype: int64


In [11]:
import plotly.graph_objects as go

# 1. Ensure data is in geographic lat/lon (EPSG:4326)
cu_geo = cu_outliers_gdf.to_crs("EPSG:4326")
deposits_geo = deposits_gdf.to_crs("EPSG:4326")

# 2. Initialize the figure
fig = go.Figure()

# 3. Add Copper Outliers (> 1000 ppm) as Blue Points
fig.add_trace(go.Scattermap(
    lat=cu_geo.geometry.y,
    lon=cu_geo.geometry.x,
    mode='markers',
    marker=dict(size=8, color='blue', opacity=0.6),
    name="Cu > 1000 ppm"
))

# 4. Add Known Deposits as Red Points
fig.add_trace(go.Scattermap(
    lat=deposits_geo.geometry.y,
    lon=deposits_geo.geometry.x,
    mode='markers',
    marker=dict(size=12, color='red'),
    name="Deposits"
))

# 5. Update map layout and style
fig.update_layout(
    map=dict(
        style="carto-positron",
        zoom=6,
        center=dict(
            lat=cu_geo.geometry.y.mean(), 
            lon=cu_geo.geometry.x.mean()
        )
    ),
    title="Cu Outliers vs Known Deposits",
    margin=dict(r=0, t=40, l=0, b=0),
    height=750
)

fig.show()

In [12]:
print(f"magnetic raster shape: {mag_az}")

magnetic raster shape: <xarray.DataArray (band: 1, y: 171, x: 174)> Size: 119kB
[29754 values with dtype=float32]
Coordinates:
  * band         (band) int64 8B 1
  * y            (y) float64 1kB 37.0 36.97 36.93 36.9 ... 31.4 31.37 31.33
  * x            (x) float64 1kB 245.2 245.2 245.3 245.3 ... 250.9 250.9 251.0
    spatial_ref  int64 8B 0
Attributes:
    AREA_OR_POINT:  Area
    _FillValue:     -3.4028235e+38
    scale_factor:   1.0
    add_offset:     0.0


In [13]:
import fiona
import geopandas as gpd

# Path to the .gdb folder you see in your directory
gdb_path = (
    r"../data/raw/USGS_StateGeologicMapCompilation_ver1.1.gdb"
)

# 1. Print all available layers inside the geodatabase
print(fiona.listlayers(gdb_path))

# Load the geology layer
sgmc_geology = gpd.read_file(gdb_path, layer="SGMC_Geology")

# INSPECT FIRST — always
print("Shape:", sgmc_geology.shape)
print("CRS:", sgmc_geology.crs)
print("Columns:", sgmc_geology.columns.tolist())
print(sgmc_geology[['STATE', 'SGMC_LABEL', 'UNIT_NAME', 'AGE_MIN', 'AGE_MAX', 'GENERALIZED_LITH', 'MAJOR1']].head() 
    if 'STATE' in sgmc_geology.columns else sgmc_geology.head())


['SGMC_Points', 'SGMC_Geology', 'SGMC_Structure', 'Age', 'Lithology', 'References', 'Unit_Ref_Link', 'Units']


c:\Users\HP\miniconda3\envs\arizona-porphyry\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


Shape: (313732, 29)
CRS: PROJCS["USA_Contiguous_Albers_Equal_Area_Conic_USGS_version",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","102039"]]
Columns: ['STATE', 'ORIG_LABEL', 'SGMC_LABEL', 'UNIT_LINK', 'UNIT_NAME', 'AGE_MIN', 'AGE_MAX', 'MAJOR1', 'MAJOR2', 'MAJOR3', 'MINOR1', 'MINOR2', 'MINOR3', 'MINOR4', 'MINOR5', 'INCIDENTAL', 'INDETERMINATE', 'REF_ID', 'REFERENCE', 'GENERALIZED_LITH', 'DIGITAL_URL', 'NGMDB1', 'NGMDB2

In [14]:
sgmc_az = sgmc_geology[sgmc_geology["STATE"].str.upper().str.strip() == "AZ"].copy()
print(f"Filtered: {len(sgmc_geology)} polygons → {len(sgmc_az)} AZ polygons")

Filtered: 313732 polygons → 4102 AZ polygons


In [15]:
# IF there's a STATE column - load the sgmc structure layer
sgmc_structure = gpd.read_file(gdb_path, layer ="SGMC_Structure")
print(sgmc_structure.columns.tolist())
print(sgmc_structure.head())
sgmc_structure_az = sgmc_structure[sgmc_structure["STATE"].str.upper() == "AZ"].copy()

['STATE', 'DESCRIPTION', 'MISC', 'REF_ID', 'REFERENCE', 'DIGITAL_URL', 'NGMDB1', 'NGMDB2', 'NGMDB3', 'Shape_Length', 'RuleID', 'Override', 'geometry']
  STATE       DESCRIPTION                                               MISC  \
0    AL  Contact, certain                 Devonian Line - Chattanooga Shale.   
1    AL  Contact, certain  Devonian Line - Chattanooga Shale and Frog Mou...   
2    AL  Contact, certain           Devonian Line - Frog Mountain Sandstone.   
3    AL  Contact, certain                 Devonian Line - Chattanooga Shale.   
4    AL  Contact, certain                 Devonian Line - Chattanooga Shale.   

  REF_ID                                          REFERENCE  \
0  AL004  Geological Survey of Alabama, 2006, GSA Specia...   
1  AL004  Geological Survey of Alabama, 2006, GSA Specia...   
2  AL004  Geological Survey of Alabama, 2006, GSA Specia...   
3  AL004  Geological Survey of Alabama, 2006, GSA Specia...   
4  AL004  Geological Survey of Alabama, 2006, GSA Spe

In [16]:
sgmc_az.to_file("../data/processed/sgmc_geology_az.gpkg", layer="geology", driver="GPKG")
sgmc_structure_az.to_file("../data/processed/sgmc_structure_az.gpkg", layer="structure", driver="GPKG")
print("Saved SGMC AZ subsets.")

Saved SGMC AZ subsets.


In [17]:
coverage.to_csv("../reports/coverage_table.csv")

In [18]:
print("Detection limits (unique LOD values among censored samples):\n")

for el in elements:
    censored_vals = geochem.loc[geochem[f"{el}_censored"] == True, f"{el}_lod"].dropna()
    unique_lods = sorted(censored_vals.unique())
    print(f"{el}: {len(unique_lods)} distinct LODs")
    print(f"     min={unique_lods[0]}, max={unique_lods[-1]}")
    print(f"     values: {unique_lods[:10]}{'...' if len(unique_lods) > 10 else ''}")
    print()


Detection limits (unique LOD values among censored samples):

Cu: 2 distinct LODs
     min=2.0, max=5.0
     values: [np.float64(2.0), np.float64(5.0)]

Mo: 8 distinct LODs
     min=1.0, max=10.0
     values: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(10.0)]

As: 12 distinct LODs
     min=0.6, max=1000.0
     values: [np.float64(0.6), np.float64(5.0), np.float64(10.0), np.float64(30.0), np.float64(40.0), np.float64(70.0), np.float64(200.0), np.float64(300.0), np.float64(400.0), np.float64(500.0)]...

Pb: 3 distinct LODs
     min=4.0, max=20.0
     values: [np.float64(4.0), np.float64(10.0), np.float64(20.0)]

Zn: 3 distinct LODs
     min=200.0, max=500.0
     values: [np.float64(200.0), np.float64(300.0), np.float64(500.0)]

Au: 28 distinct LODs
     min=0.001, max=50.0
     values: [np.float64(0.001), np.float64(0.002), np.float64(0.003), np.float64(0.004), np.float64(0.006), np.float64(0.008), np.

In [19]:
# For each element, show the LOD distribution
for el in ["Au", "As", "Zn", "Mo", "Cu", "Pb"]:
    lods = geochem[f"{el}_lod"].dropna()
    if len(lods) == 0:
        continue
    
    print(f"\n=== {el} ===")
    print(f"Total censored samples: {len(lods)}")
    
    # Count how many samples have each LOD
    lod_counts = lods.value_counts().sort_index()
    print(lod_counts.to_string())
    print()


=== Au ===
Total censored samples: 13126
Au_lod
0.001       54
0.002     2326
0.003        3
0.004      433
0.006        1
0.008       26
0.010        1
0.020      297
0.040       15
0.050     1920
0.060        6
0.070        2
0.080        2
0.090        1
0.100      182
0.130        2
0.160        1
0.170        1
0.250      100
0.290        1
0.500        1
0.550        1
0.830        1
8.000      844
10.000    5727
15.000     583
20.000     594
50.000       1


=== As ===
Total censored samples: 12431
As_lod
0.6          2
5.0         31
10.0      2194
30.0         1
40.0         1
70.0         1
200.0     9245
300.0       39
400.0        1
500.0      110
700.0      559
1000.0     247


=== Zn ===
Total censored samples: 9531
Zn_lod
200.0    9412
300.0     115
500.0       4


=== Mo ===
Total censored samples: 12508
Mo_lod
1.0        5
2.0     3290
3.0      298
5.0     8909
6.0        1
7.0        2
8.0        1
10.0       2


=== Cu ===
Total censored samples: 162
Cu_lod
2.0     

In [20]:
import plotly.graph_objects as go

# 1. Filter to only low-LOD samples for Gold (Au)
au_low_lod = geochem[(geochem["Au_lod"] < 1.0) | (geochem["Au_imputed"].notna() & (geochem["Au_lod"].isna()))
].copy()

print(f"Au low-LOD samples: {len(au_low_lod)} / {len(geochem)}")

# 2. Initialize the Plotly figure
fig = go.Figure()

# 3. Add Low-LOD Au samples as red points
fig.add_trace(go.Scattermap(
    lat=au_low_lod["lat"],
    lon=au_low_lod["lon"],
    mode='markers',
    marker=dict(size=4, color='red', opacity=0.5),
    name="Low-LOD Au"
))

# 4. Update map layout and style
fig.update_layout(
    map=dict(
        style="carto-positron",
        zoom=6,
        center=dict(
            lat=au_low_lod["lat"].mean(), 
            lon=au_low_lod["lon"].mean()
        )
    ),
    title=f"Low-LOD Au samples ({len(au_low_lod)} of {len(geochem)})",
    margin=dict(r=0, t=40, l=0, b=0),
    height=750
)

Au low-LOD samples: 7054 / 15052


In [21]:
# Threshold for "anomalous Au" — 10 ppb is a standard geochemical anomaly threshold
AU_THRESHOLD = 10  # ppb

# Classify each sample
geochem["Au_above_thresh"] = pd.NA  # default unknown

# Rule 1: If measured and above threshold → True
measured_above = geochem["Au"].notna() & (geochem["Au"] > AU_THRESHOLD)
geochem.loc[measured_above, "Au_above_thresh"] = True

# Rule 2: If measured and below threshold → False  
measured_below = geochem["Au"].notna() & (geochem["Au"] <= AU_THRESHOLD)
geochem.loc[measured_below, "Au_above_thresh"] = False

# Rule 3: If censored and LOD > threshold → UNKNOWN (we can't rule out it's above)
# (This is the case where LOD = 50 ppb and threshold = 10 ppb → sample could be 20 ppb!)
censored_unclear = (geochem["Au_censored"] == True) & (geochem["Au_lod"] > AU_THRESHOLD)
# → leave as pd.NA (unknown)

# Rule 4: If censored and LOD <= threshold → False (definitively below)
censored_below = (geochem["Au_censored"] == True) & (geochem["Au_lod"] <= AU_THRESHOLD)
geochem.loc[censored_below, "Au_above_thresh"] = False

# Check the classification
print(geochem["Au_above_thresh"].value_counts(dropna=False))

Au_above_thresh
False    13616
<NA>      1427
True         9
Name: count, dtype: int64


In [22]:
thresholds = {
    "Cu": 50,    # ppm — moderately enriched
    "Pb": 30,    # ppm — above background  
    "Mo": 5,     # ppm — above background
    "As": 20,    # ppm — above background
}

for el, thresh in thresholds.items():
    col = f"{el}_above_thresh"
    geochem[col] = pd.NA

    # Measured (raw value present, not censored)
    measured = geochem[f"{el}"].notna()
    # Measured above threshold
    mask = measured & (geochem[f"{el}"] > thresh)
    geochem.loc[mask, col] = True
    
    # Measured below threshold
    mask = measured & (geochem[f"{el}"] <= thresh)
    geochem.loc[mask, col] = False
    
    # Censored below threshold (definitively below)
    mask = (geochem[f"{el}_censored"] == True) & (geochem[f"{el}_lod"] <= thresh)
    geochem.loc[mask, col] = False
    
    # Censored above threshold → leave as NA (unknown)


In [23]:
print("Above-threshold classification summary:\n")
for el in ["Cu", "Pb", "Mo", "As", "Au"]:
    col = geochem[f"{el}_above_thresh"]
    counts = col.value_counts(dropna=False)
    total = len(col)
    known = counts.get(True, 0) + counts.get(False, 0)
    print(f"{el}:")
    print(f"  True:  {counts.get(True, 0):,}  ({100*counts.get(True, 0)/known:.1f}% of known)")
    print(f"  False: {counts.get(False, 0):,}")
    print(f"  NA:    {counts.get(pd.NA, 0):,}")
    print()

Above-threshold classification summary:

Cu:
  True:  3,221  (21.8% of known)
  False: 11,547
  NA:    284

Pb:
  True:  4,238  (28.7% of known)
  False: 10,530
  NA:    284

Mo:
  True:  832  (5.6% of known)
  False: 13,930
  NA:    290

As:
  True:  462  (10.0% of known)
  False: 4,146
  NA:    10,444

Au:
  True:  9  (0.1% of known)
  False: 13,616
  NA:    1,427



In [24]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

plot_elements = ["Cu", "Mo", "As", "Pb", "Zn", "Au"]

# Create a 2x3 subplot grid in Plotly
fig = make_subplots(
    rows=2, 
    cols=3, 
    subplot_titles=plot_elements,
    horizontal_spacing=0.08,
    vertical_spacing=0.12
)

# Keep track of grid positions for the loop
row, col = 1, 1

for i, el in enumerate(plot_elements):
    # 1. Measured values (real)
    measured = geochem.loc[
        geochem[f"{el}_imputed"].notna() & (geochem[f"{el}_censored"] != True), 
        f"{el}_imputed"
    ]
    
    # 2. Imputed (censored → LOD/2)
    imputed = geochem.loc[
        geochem[f"{el}_censored"] == True, 
        f"{el}_imputed"
    ]
    
    # Log transform for display
    measured_log = np.log10(measured[measured > 0])
    imputed_log = np.log10(imputed[imputed > 0])
    
    # Add Measured histogram trace
    fig.add_trace(
        go.Histogram(
            x=measured_log, 
            nbinsx=50, 
            name=f"{el} Measured", 
            marker_color="steelblue", 
            opacity=0.7,
            legendgroup="Measured",
            showlegend=(i == 0) # Only show legend once for the whole group
        ), 
        row=row, col=col
    )
    
    # Add Imputed histogram trace
    fig.add_trace(
        go.Histogram(
            x=imputed_log, 
            nbinsx=50, 
            name=f"{el} Imputed", 
            marker_color="orange", 
            opacity=0.7,
            legendgroup="Imputed",
            showlegend=(i == 0) # Only show legend once for the whole group
        ), 
        row=row, col=col
    )
    
    # Update axis labels for each subplot
    fig.update_xaxes(title_text="log10(ppm)", row=row, col=col)
    fig.update_yaxes(title_text="Count", row=row, col=col)
    
    # Advance grid position (2 rows, 3 columns)
    col += 1
    if col > 3:
        col = 1
        row += 1

# Update overall layout to match your desired dimensions and style
fig.update_layout(
    barmode='overlay',
    height=800,
    title_text="Geochemical Element Distributions (Measured vs. Imputed LOD/2)",
    template="plotly_white"
)

fig.show()

# Optional: If you also want to save a static copy to your reports folder:
# fig.write_image("../reports/figures/geochem_histograms.png", width=1200, height=800, scale=2)

In [25]:
# Compare median values against known background ranges - Units consistency check
for el in ["Cu","Mo","As","Pb","Zn","Au"]:
    print(f"{el}: median={geochem[f'{el}_imputed'].median():.3f}, "
        f"p95={geochem[f'{el}_imputed'].quantile(0.95):.3f}")

Cu: median=30.000, p95=150.000
Mo: median=2.500, p95=7.000
As: median=100.000, p95=350.000
Pb: median=28.000, p95=100.000
Zn: median=100.000, p95=200.000
Au: median=4.000, p95=7.500


In [26]:
#Duplicate location check
dupes = geochem.groupby(["lat","lon"]).size().sort_values(ascending=False)
print(f"Unique locations: {dupes.shape[0]}")
print(f"Locations with >1 sample: {(dupes > 1).sum()}")
print(dupes.head(10))

Unique locations: 13371
Locations with >1 sample: 936
lat        lon        
33.381167  -110.784554    81
33.378945  -110.830666    64
31.500095  -110.750641    49
33.556163  -110.804000    36
33.334779  -110.820388    33
32.905610  -111.884856    32
31.616757  -110.867309    32
31.500092  -111.000644    26
33.587552  -110.804000    21
31.500094  -110.883976    18
dtype: int64


In [27]:
geochem.to_csv("../data/processed/geochem_clean.csv", index=False)
print(f"Saved: geochem_clean.csv ({len(geochem)} rows, {len(geochem.columns)} cols)")

Saved: geochem_clean.csv (15052 rows, 32 cols)


# SECTION D

In [28]:
geochem_gdf = gpd.GeoDataFrame(
    geochem,
    geometry=gpd.points_from_xy(geochem["lon"], geochem["lat"]),
    crs="EPSG:4326"
).to_crs("EPSG:32612")

In [29]:
samples_with_cell = gpd.sjoin(
    geochem_gdf,
    grid[["cell_id", "deposit", "geometry"]],
    how="left",
    predicate="within"
)

In [30]:
samples_per_cell = samples_with_cell.groupby("cell_id").size().rename("n_samples")

# Merge back onto the grid (so cells with 0 samples are preserved)
grid_d = grid.merge(samples_per_cell, on="cell_id", how="left")
grid_d["n_samples"] = grid_d["n_samples"].fillna(0).astype(int)

In [31]:
n_cells = len(grid_d)
n_cells_with_samples = (grid_d["n_samples"] > 0).sum()
n_cells_zero = (grid_d["n_samples"] == 0).sum()

print(f"Total cells:            {n_cells}")
print(f"Cells with ≥1 sample:   {n_cells_with_samples}  ({100*n_cells_with_samples/n_cells:.1f}%)")
print(f"Cells with 0 samples:   {n_cells_zero}  ({100*n_cells_zero/n_cells:.1f}%)")
print()
print("Samples per cell (cells with ≥1):")
print(grid_d.loc[grid_d["n_samples"] > 0, "n_samples"].describe())

# Deposit cell coverage
dep = grid_d[grid_d["deposit"] == 1]
nodep = grid_d[grid_d["deposit"] == 0]

print()
print(f"Deposit cells with ≥1 sample:    {(dep['n_samples']>0).sum()} / {len(dep)}")
print(f"Non-deposit cells with ≥1 sample: {(nodep['n_samples']>0).sum()} / {len(nodep)}")

Total cells:            3064
Cells with ≥1 sample:   1545  (50.4%)
Cells with 0 samples:   1519  (49.6%)

Samples per cell (cells with ≥1):
count    1545.000000
mean        9.741100
std        16.008265
min         1.000000
25%         1.000000
50%         4.000000
75%        12.000000
max       180.000000
Name: n_samples, dtype: float64

Deposit cells with ≥1 sample:    30 / 37
Non-deposit cells with ≥1 sample: 1515 / 3027


In [38]:
import plotly.express as px
import plotly.graph_objects as go

# 1. Convert grid_d to geographic lat/lon (EPSG:4326) FIRST
grid_geo = grid_d.to_crs("EPSG:4326")
centroids = grid_geo.geometry.centroid

grid_plot_df = grid_geo.copy()
grid_plot_df['lon'] = centroids.x
grid_plot_df['lat'] = centroids.y

# 2. Ensure deposits are also in EPSG:4326
deposits_geo = deposits_gdf.to_crs("EPSG:4326")

# 3. Plot with px.scatter_map
fig = px.scatter_map(
    grid_plot_df,
    lat="lat",
    lon="lon",
    color="n_samples",
    color_continuous_scale="YlOrRd",
    zoom=6,
    center=dict(lat=grid_plot_df['lat'].mean(), lon=grid_plot_df['lon'].mean()),
    title="Geochem sample density per 10 km cell",
    height=750
)

# 4. Add deposits overlay
fig.add_trace(go.Scattermap(
    lat=deposits_geo.geometry.y,
    lon=deposits_geo.geometry.x,
    mode='markers',
    marker=dict(size=10, color='blue'),
    name="Deposits"
))

fig.update_layout(
    map=dict(style="carto-positron"),
    margin=dict(r=0, t=40, l=0, b=0)
)

fig.show()

C:\Users\HP\AppData\Local\Temp\ipykernel_25768\1780695826.py:6: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = grid_geo.geometry.centroid


In [36]:
import plotly.express as px

# Filter cells with more than 0 samples
sample_data = grid_d.loc[grid_d["n_samples"] > 0, "n_samples"]

# Create interactive histogram with a log Y-axis
fig = px.histogram(
    x=sample_data,
    nbins=50,
    log_y=True,
    title="Distribution of samples per cell",
    labels={"x": "Samples per cell", "y": "Number of cells (log)"}
)

fig.update_layout(
    xaxis_title="Samples per cell",
    yaxis_title="Number of cells (log)",
    bargap=0.05,
    height=500
)

fig.show()

In [37]:
print("Deposit cells:")
print(dep["n_samples"].describe())
print()
print("Non-deposit cells:")
print(nodep["n_samples"].describe())
print()
print(f"Mean samples per deposit cell:     {dep['n_samples'].mean():.1f}")
print(f"Mean samples per non-deposit cell: {nodep['n_samples'].mean():.1f}")
print(f"Ratio:                             {dep['n_samples'].mean() / nodep['n_samples'].mean():.2f}")

Deposit cells:
count     37.000000
mean      20.891892
std       46.662966
min        0.000000
25%        1.000000
50%        5.000000
75%       13.000000
max      180.000000
Name: n_samples, dtype: float64

Non-deposit cells:
count    3027.000000
mean        4.716551
std        11.212082
min         0.000000
25%         0.000000
50%         1.000000
75%         4.000000
max       171.000000
Name: n_samples, dtype: float64

Mean samples per deposit cell:     20.9
Mean samples per non-deposit cell: 4.7
Ratio:                             4.43
